In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
BASE_ENV = Path().resolve().parent

In [4]:
df = pd.read_csv(BASE_ENV / 'data/raw/cicids_wednesday_2018.csv')

In [5]:
COLUMNS_TO_DROP = [
    'CWE_Flag_Count',
    'Bwd_Blk_Rate_Avg',
    'Subflow_Fwd_Byts',
    'Subflow_Fwd_Pkts',
    'Subflow_Bwd_Byts',
    'Fwd_Pkts_b_Avg',
    'Fwd_Blk_Rate_Avg',
    'Bwd_Byts_b_Avg',
    'Bwd_Pkts_b_Avg',
    'Fwd_Byts_b_Avg',
    'Fwd_Seg_Size_Avg',
    'Subflow_Bwd_Pkts',
    'SYN_Flag_Cnt',
    'Bwd_URG_Flags',
    'Fwd_URG_Flags',
    'Fwd_IAT_Mean',
    'Fwd_IAT_Std',
    'Bwd_IAT_Tot',
    'Fwd_Pkt_Len_Std',
    'Idle_Std',
    'Fwd_IAT_Min',
    'Bwd_IAT_Max',
    'Bwd_IAT_Min',
    'Pkt_Size_Avg',
    'Fwd_PSH_Flags',
    'Protocol',
    'Bwd_IAT_Mean',
    'Bwd_IAT_Std',
    'Idle_Max',
    'FIN_Flag_Cnt',
    'RST_Flag_Cnt',
    'Active_Min',
    'ECE_Flag_Cnt',
    'Idle_Mean',
    'Active_Max',
    'Idle_Min',
    'Fwd_Pkt_Len_Min',
    'Bwd_Pkt_Len_Min',
    'Bwd_PSH_Flags',
    'Pkt_Len_Min',
    'Active_Mean',
    'Active_Std'
    ]

In [ ]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('/', '_')


In [7]:
df = df.drop(columns=COLUMNS_TO_DROP)
df = df.drop(columns='Timestamp')
df = df.drop(columns='Dst_Port')

In [8]:
df = df[df['Label'] != 'Heartbleed']

In [9]:
# drop the columns with infinite and nan values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(axis=1)

In [10]:
df['Init_Fwd_Win_Byts'] = df['Init_Fwd_Win_Byts'].replace(-1, 0)
df['Init_Bwd_Win_Byts'] = df['Init_Bwd_Win_Byts'].replace(-1, 0)

negative_cols = df.select_dtypes(include=[np.number]).columns
df = df[(df[negative_cols] >= 0).all(axis=1)]

In [11]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['Label'])
test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['Label'])

In [12]:
scaler = MinMaxScaler()
train_outputs = train_df['Label']
test_outputs = test_df['Label']
val_outputs = val_df['Label']
train_df_scaled = scaler.fit_transform(train_df.drop(columns=['Label']))
test_df_scaled = scaler.transform(test_df.drop(columns=['Label']))
val_df_scaled = scaler.transform(val_df.drop(columns=['Label']))


In [13]:
encoder = LabelEncoder()
train_outputs_encoded = encoder.fit_transform(train_outputs)
test_outputs_encoded = encoder.transform(test_outputs)
val_outputs_encoded = encoder.transform(val_outputs)

In [ ]:
ART = BASE_ENV / 'artifacts'

In [ ]:
# ART = BASE_ENV / "artifacts"

# train
np.save(ART / "datasets/train/X.npy", train_df_scaled)
np.save(ART / "datasets/train/y.npy", train_outputs_encoded)

# val
np.save(ART / "datasets/val/X.npy", val_df_scaled)
np.save(ART / "datasets/val/y.npy", val_outputs_encoded)

# test
np.save(ART / "datasets/test/X.npy", test_df_scaled)
np.save(ART / "datasets/test/y.npy", test_outputs_encoded)

# scaler
joblib.dump(scaler, ART / "preprocessors/scaler.joblib")

# encoder
joblib.dump(encoder, ART / "preprocessors/encoder.joblib")

['/mnt/c/Users/hassa/OneDrive/Documents/Programming/Python/projects/heimdall/Heimdall/ai-service/IDS/artifacts/preprocessors/encoder.joblib']